In [ ]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

In [ ]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

# OPENAI_API_KEY et TAVILY_API_KEY sont lus automatiquement depuis l'environnement
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [ ]:

search_tool = TavilySearch(max_results=2)

In [ ]:
# TavilySearch renvoie un dict : les résultats sont dans la clé "results"
result = search_tool.invoke("Give me the latest news on AI")
result["results"]

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather information for a city."""
    response = requests.get(
        "https://api.weatherstack.com/current",
        params={"access_key": WEATHERSTACK_API_KEY, "query": city},
        timeout=10,
    )
    data = response.json()

    if "current" not in data:
        error = data.get("error", {}).get("info", "unknown error")
        return f"Could not fetch weather data for {city}: {error}"

    current = data["current"]
    return (
        f"City: {city}\n"
        f"Temperature: {current['temperature']}°C\n"
        f"Weather: {current['weather_descriptions'][0]}\n"
        f"Humidity: {current['humidity']}%"
    )

In [ ]:
# ==========================================
# LLM
# ==========================================
# gpt-3.5-turbo est dépassé : gpt-4o-mini est moins cher et bien meilleur pour appeler des outils
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
response = llm.invoke("Tell me a joke about AI")
response.content

In [ ]:
# ==========================================
# TOOLS
# ==========================================
tools = [search_tool, get_weather_data]

In [ ]:
# ==========================================
# CREATE AGENT
# ==========================================
# Remplace hub.pull("hwchase17/react") + create_react_agent + AgentExecutor
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the tools when you need up-to-date information.",
)

In [ ]:

# stream_mode="values" affiche chaque étape (remplace verbose=True)
question = "Find the capital of India and then find its current weather."

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

agent_response = step

In [ ]:

print(agent_response["messages"][-1].content)